# Federated Learning — Convergence Analysis & Ablation Study

This notebook runs the proposed **SimpleNetAugDR-3** model under **FedAvg** for
**20 communication rounds** to study convergence and justify the number of rounds
adopted in the paper (Table VII / Figure 16).

**Data split (proper three-way).** `train/` is split (stratified) into a **client
training pool** and a **validation** set; the pool is partitioned across the clients.
`valid/` is kept as a **held-out test set**. The convergence study below monitors the
global model on the **validation** set each round (choosing the number of rounds is a
selection decision, so it must not use test). The adopted round count is then confirmed
once on the held-out test set at the end.

## 1. Imports

In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall, AUC

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix)

import warnings
warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)

In [ ]:
# ----------------------------- Configuration -------------------------------
RANDOM_SEED = 123
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); tf.random.set_seed(RANDOM_SEED)

TRAIN_DIR = './new-plant-diseases-dataset/train/'
VALID_DIR = './new-plant-diseases-dataset/valid/'

NUM_CLIENTS   = 4
NUM_ROUNDS    = 20        # convergence study horizon (reviewer request: 10-20 rounds)
ADOPTED_ROUND = 5         # round count adopted in the paper
LOCAL_EPOCHS  = 3         # local epochs per client per round
BATCH_SIZE    = 16
LR            = 0.001
VAL_SPLIT     = 0.2       # fraction of train/ held out as validation
IMG_SIZE      = (128, 128)
NUM_CLASSES   = 10

OUT_DIR = 'fl_convergence_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load & preprocess data (train / validation / test)

`train/` is split (stratified) into a **client pool** and a **validation** set; `valid/` is the **held-out test set**.

In [ ]:
def load_data(dir_path, img_size=IMG_SIZE):
    X, Y, i = [], [], 0
    for path in tqdm(sorted(os.listdir(dir_path)), desc=os.path.basename(dir_path.rstrip('/'))):
        if path.startswith('.') or 'Tomato__' not in path:
            continue
        sub = os.path.join(dir_path, path)
        for f in os.listdir(sub):
            fp = os.path.join(sub, f)
            if not f.startswith('.') and os.path.isfile(fp):
                img = cv2.imread(fp)
                img = Image.fromarray(img, 'RGB').resize(img_size)
                X.append(np.array(img)); Y.append(i)
        i += 1
    print(f'{len(X)} images loaded from {dir_path}')
    return np.array(X), np.array(Y)

In [ ]:
from tensorflow.keras.utils import to_categorical, normalize

# Client pool + validation (from train/), held-out test (from valid/).
X_all, y_all = load_data(TRAIN_DIR)
X_te,  y_te  = load_data(VALID_DIR)

X_pool, X_val, y_pool, y_val = train_test_split(
    X_all, y_all, test_size=VAL_SPLIT, stratify=y_all, random_state=RANDOM_SEED)

X_pool = normalize(X_pool.astype(np.float32), axis=1)
X_val  = normalize(X_val.astype(np.float32),  axis=1)
X_test = normalize(X_te.astype(np.float32),   axis=1)

y_val_cat  = to_categorical(y_val, NUM_CLASSES)
y_test_cat = to_categorical(y_te,  NUM_CLASSES)
print(f"Client pool: {X_pool.shape} | Val: {X_val.shape} | Test (held-out): {X_test.shape}")

## 3. Split the training pool across 4 clients (stratified)

In [ ]:
def make_clients(X, y_int, num_clients, seed=RANDOM_SEED):
    """Stratified partition into num_clients balanced shards (IID setting)."""
    rng = np.random.default_rng(seed)
    client_idx = [[] for _ in range(num_clients)]
    for c in np.unique(y_int):
        idx = np.where(y_int == c)[0]; rng.shuffle(idx)
        for k, part in enumerate(np.array_split(idx, num_clients)):
            client_idx[k].extend(part.tolist())
    clients = []
    for k in range(num_clients):
        ix = np.array(client_idx[k]); rng.shuffle(ix)
        clients.append((X[ix], to_categorical(y_int[ix], NUM_CLASSES)))
    return clients


client_data = make_clients(X_pool, y_pool, NUM_CLIENTS)
for i, (xc, _) in enumerate(client_data):
    print(f"Client {i+1}: {xc.shape[0]} samples")

## 4. Model definition (SimpleNetAugDR-3)

In [ ]:
def create_model(num_classes=NUM_CLASSES, lr=LR):
    model = Sequential([
        Conv2D(64, (4, 4), activation='relu', input_shape=(128, 128, 3)),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (4, 4), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), activation='relu'),   # 64/64/64 -> matches Figure 2 & benchmark NB01
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])
    model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=lr),
                  metrics=['accuracy', Precision(name='precision'),
                           Recall(name='recall'), AUC(name='auc')])
    return model


create_model().summary()

## 5. Federated learning — 20-round convergence study

Each round, every client trains locally for `LOCAL_EPOCHS` from the current global
weights; the server aggregates with **weighted FedAvg** ($w_{t+1}=\sum_k \tfrac{n_k}{N}w_k$).
The global model is evaluated on the **validation** set after each round (these are the
metrics analysed below). The global model at the adopted round (`ADOPTED_ROUND`) is saved
for the held-out-test confirmation in Section 9.

In [ ]:
def fedavg(local_weights, sizes):
    total = float(sum(sizes))
    return [sum((sizes[k] / total) * lw[layer] for k, lw in enumerate(local_weights))
            for layer in range(len(local_weights[0]))]


def evaluate_round(model, X, y_cat):
    y_pred = np.argmax(model.predict(X, batch_size=BATCH_SIZE, verbose=0), axis=1)
    y_true = np.argmax(y_cat, axis=1)
    loss = model.evaluate(X, y_cat, verbose=0)[0]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    return {'loss': round(loss, 4),
            'accuracy':  round(accuracy_score(y_true, y_pred) * 100, 2),
            'precision': round(prec * 100, 2),
            'recall':    round(rec * 100, 2),
            'f1_score':  round(f1 * 100, 2)}


tf.keras.utils.set_random_seed(RANDOM_SEED)
global_model = create_model()
sizes = [len(xc) for xc, _ in client_data]
adopted_path = os.path.join(OUT_DIR, f'global_round{ADOPTED_ROUND}.keras')

results = {k: [] for k in ['round', 'loss', 'accuracy', 'precision', 'recall', 'f1_score']}
print(f"FL convergence study: {NUM_ROUNDS} rounds, {NUM_CLIENTS} clients, "
      f"{LOCAL_EPOCHS} local epochs each\n" + "=" * 70)

for round_num in range(1, NUM_ROUNDS + 1):
    global_w = global_model.get_weights()
    local_weights = []
    for client_id in range(NUM_CLIENTS):
        local = create_model()
        local.set_weights(global_w)
        X_c, y_c = client_data[client_id]
        local.fit(X_c, y_c, batch_size=BATCH_SIZE, epochs=LOCAL_EPOCHS, verbose=0)
        local_weights.append(local.get_weights())
        del local; gc.collect()

    global_model.set_weights(fedavg(local_weights, sizes))      # weighted FedAvg

    m = evaluate_round(global_model, X_val, y_val_cat)           # monitor on validation
    results['round'].append(round_num)
    for k in ['loss', 'accuracy', 'precision', 'recall', 'f1_score']:
        results[k].append(m[k])
    print(f"Round {round_num:2d}/{NUM_ROUNDS} | val acc {m['accuracy']:.2f}% | "
          f"loss {m['loss']:.4f} | P {m['precision']:.2f} R {m['recall']:.2f} F1 {m['f1_score']:.2f}")

    if round_num == ADOPTED_ROUND:
        global_model.save(adopted_path)

print("\n" + "=" * 70 + "\nConvergence study complete.")

## 6. Save results to CSV

In [ ]:
df_results = pd.DataFrame(results)
csv_path = os.path.join(OUT_DIR, 'fl_convergence_results_20rounds.csv')
df_results.to_csv(csv_path, index=False)
print("Saved ->", csv_path, "\n")
print(df_results.to_string(index=False))

## 7. Convergence analysis — statistical justification for the adopted round count

All metrics below are on the **validation** set.

In [ ]:
acc = df_results['accuracy'].values
delta_acc = np.abs(np.diff(acc))   # |change| between consecutive rounds

CONVERGENCE_THRESHOLD = 0.10  # < 0.10 pp change = converged

converged_at = None
for i, d in enumerate(delta_acc):
    if d < CONVERGENCE_THRESHOLD:
        # Check if stays converged for 2 more rounds
        if i + 2 < len(delta_acc) and delta_acc[i+1] < CONVERGENCE_THRESHOLD:
            converged_at = i + 1   # round index (1-based)
            break

print("=" * 60)
print("CONVERGENCE ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nAccuracy at each round:")
for r, a in zip(df_results['round'], df_results['accuracy']):
    marker = " ◄ CONVERGED" if r == converged_at else ""
    print(f"  Round {r:2d}: {a:.2f}%{marker}")

print(f"\nRound-to-round accuracy change (|Δ|):")
for i, d in enumerate(delta_acc):
    print(f"  Round {i+1} → {i+2}: {d:.4f} pp")

print(f"\n► Convergence threshold: < {CONVERGENCE_THRESHOLD} pp")
if converged_at:
    print(f"► Model converges at round: {converged_at}")
    print(f"► Accuracy at round 5:  {acc[4]:.2f}%")
    print(f"► Accuracy at round 20: {acc[-1]:.2f}%")
    print(f"► Gain from round 5→20: {acc[-1]-acc[4]:.4f} pp")
    print(f"\n► CONCLUSION: 5 rounds is sufficient — "
          f"convergence occurs at round {converged_at} "
          f"and additional rounds yield negligible improvement.")
else:
    print("► Model is still converging beyond 20 rounds — consider more rounds.")

## 8. Publication-quality figures (validation metrics over rounds)

In [ ]:
rounds   = df_results['round'].values
loss     = df_results['loss'].values
accuracy = df_results['accuracy'].values
precision= df_results['precision'].values
recall   = df_results['recall'].values
f1       = df_results['f1_score'].values
delta_acc = np.abs(np.diff(accuracy))

COLORS = {
    'accuracy':  '#2196F3',
    'precision': '#4CAF50',
    'recall':    '#FF9800',
    'f1':        '#9C27B0',
    'loss':      '#F44336',
    'delta':     '#607D8B',
    'ref5':      '#E53935',
}

plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Full learning curves (Accuracy + Loss  |  Precision/Recall/F1)
# ════════════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle(
    'FL Learning Curves over 20 Rounds — SimpleNetAugDR-3',
    fontsize=14, fontweight='bold', y=1.02
)

# Left: Accuracy & Loss
ax1 = axes[0]
ax1b = ax1.twinx()
l1, = ax1.plot(rounds, accuracy,  'o-', color=COLORS['accuracy'],
               linewidth=2, markersize=6, label='Accuracy (%)')
l2, = ax1b.plot(rounds, loss,     's--', color=COLORS['loss'],
                linewidth=2, markersize=5, label='Loss')
ax1.axvline(x=5, color=COLORS['ref5'], linestyle=':', linewidth=2,
            label='Round 5 (paper)')
ax1.set_xlabel('FL Round')
ax1.set_ylabel('Accuracy (%)', color=COLORS['accuracy'])
ax1b.set_ylabel('Loss',        color=COLORS['loss'])
ax1.set_xticks(rounds)
ax1.set_title('Accuracy & Loss')
ax1.grid(True, alpha=0.3)
lines = [l1, l2,
         plt.Line2D([0],[0], color=COLORS['ref5'], linestyle=':', linewidth=2,
                    label='Round 5 (paper)')]
ax1.legend(handles=lines, loc='lower right')

# Right: Precision / Recall / F1
ax2 = axes[1]
ax2.plot(rounds, precision, 'o-', color=COLORS['precision'],
         linewidth=2, markersize=6, label='Precision (%)')
ax2.plot(rounds, recall,    's-', color=COLORS['recall'],
         linewidth=2, markersize=6, label='Recall (%)')
ax2.plot(rounds, f1,        '^-', color=COLORS['f1'],
         linewidth=2, markersize=6, label='F1-Score (%)')
ax2.axvline(x=5, color=COLORS['ref5'], linestyle=':', linewidth=2,
            label='Round 5 (paper)')
ax2.set_xlabel('FL Round')
ax2.set_ylabel('Score (%)')
ax2.set_xticks(rounds)
ax2.set_title('Precision, Recall & F1-Score')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_learning_curves_20rounds.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved: fig_learning_curves_20rounds.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Convergence Detection (delta accuracy per round)
# ════════════════════════════════════════════════════════════════════════════
fig2, ax = plt.subplots(figsize=(10, 4))

delta_rounds = rounds[1:]   # delta is between round i and i+1
bars = ax.bar(delta_rounds, delta_acc, color=COLORS['delta'],
              alpha=0.75, edgecolor='white', linewidth=0.5)

# Highlight bars above threshold
for bar, d in zip(bars, delta_acc):
    if d >= CONVERGENCE_THRESHOLD:
        bar.set_color('#EF5350')
        bar.set_alpha(0.9)

ax.axhline(y=CONVERGENCE_THRESHOLD, color='black', linestyle='--',
           linewidth=1.5, label=f'Convergence threshold ({CONVERGENCE_THRESHOLD} pp)')
ax.axvline(x=5.5, color=COLORS['ref5'], linestyle=':', linewidth=2,
           label='After round 5 (paper choice)')

ax.set_xlabel('FL Round')
ax.set_ylabel('|ΔAccuracy| (pp)')
ax.set_title('Round-to-Round Accuracy Change — Convergence Detection',
             fontweight='bold')
ax.set_xticks(delta_rounds)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Annotate each bar
for bar, d in zip(bars, delta_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{d:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('fig_convergence_detection.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved: fig_convergence_detection.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Accuracy bar chart across all 20 rounds
# ════════════════════════════════════════════════════════════════════════════
fig3, ax = plt.subplots(figsize=(13, 5))

bar_colors = [
    COLORS['ref5'] if r <= 5 else COLORS['accuracy'] for r in rounds
]
bars = ax.bar(rounds, accuracy, color=bar_colors, edgecolor='white',
              linewidth=0.5, alpha=0.85)

# Value labels
for bar, a in zip(bars, accuracy):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{a:.2f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.axvline(x=5.5, color='black', linestyle='--', linewidth=1.5,
           label='5-round boundary (paper choice)')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS['ref5'],      label='Rounds 1–5 (paper)'),
    Patch(facecolor=COLORS['accuracy'],  label='Rounds 6–20 (extended)'),
    plt.Line2D([0],[0], color='black', linestyle='--', label='5-round boundary'),
]
ax.legend(handles=legend_elements, loc='lower right')

ax.set_xlabel('FL Round')
ax.set_ylabel('Global Accuracy (%)')
ax.set_title('Global Model Accuracy per FL Round — 20-Round Ablation Study',
             fontweight='bold')
ax.set_xticks(rounds)
ax.set_ylim(min(accuracy) - 1, max(accuracy) + 2)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_accuracy_barchart_20rounds.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved: fig_accuracy_barchart_20rounds.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Heatmap of all metrics across all rounds
# ════════════════════════════════════════════════════════════════════════════
heatmap_data = df_results[['accuracy','precision','recall','f1_score','loss']].T
heatmap_data.columns = [f'R{r}' for r in rounds]
heatmap_data.index   = ['Accuracy (%)', 'Precision (%)', 'Recall (%)',
                         'F1-Score (%)', 'Loss']

fig4, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    heatmap_data,
    ax=ax,
    annot=True, fmt='.2f',
    cmap='YlOrRd_r',
    linewidths=0.5,
    annot_kws={'size': 8},
    cbar_kws={'label': 'Value'}
)
ax.set_title('FL Metrics Heatmap — All 20 Rounds', fontweight='bold')
ax.set_xlabel('FL Round')

# Highlight round 5 column
ax.add_patch(plt.Rectangle((4, 0), 1, 5, fill=False,
                             edgecolor=COLORS['ref5'], lw=3, clip_on=False))
ax.text(4.5, -0.3, 'Round 5\n(paper)', ha='center', va='top',
        color=COLORS['ref5'], fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_metrics_heatmap_20rounds.png', dpi=180, bbox_inches='tight')
plt.show()
print("Saved: fig_metrics_heatmap_20rounds.png")

## 9. Summary table & held-out-test confirmation

In [ ]:
print("=" * 70)
print("SUMMARY TABLE — FL GLOBAL MODEL METRICS ACROSS 20 ROUNDS")
print("=" * 70)
print(df_results.to_string(index=False))

# Key stats for rebuttal
acc5  = df_results.loc[df_results['round'] == 5,  'accuracy'].values[0]
acc10 = df_results.loc[df_results['round'] == 10, 'accuracy'].values[0]
acc20 = df_results.loc[df_results['round'] == 20, 'accuracy'].values[0]

print("\n" + "=" * 70)
print("KEY STATISTICS FOR REBUTTAL")
print("=" * 70)
print(f"  Accuracy @ Round  5 : {acc5:.2f}%")
print(f"  Accuracy @ Round 10 : {acc10:.2f}%")
print(f"  Accuracy @ Round 20 : {acc20:.2f}%")
print(f"  Gain Round  5 → 10  : {acc10 - acc5:+.4f} pp")
print(f"  Gain Round  5 → 20  : {acc20 - acc5:+.4f} pp")
print(f"  Gain Round 10 → 20  : {acc20 - acc10:+.4f} pp")

max_delta_after5 = np.max(delta_acc[4:])   # max change after round 5
print(f"\n  Max |Δ accuracy| after round 5 : {max_delta_after5:.4f} pp")
print(f"  Convergence threshold           : {CONVERGENCE_THRESHOLD} pp")
if max_delta_after5 < CONVERGENCE_THRESHOLD:
    print("\n  ✓ All changes after round 5 are below the convergence threshold.")
    print("  ✓ This statistically justifies the choice of 5 rounds in the paper.")
else:
    print("\n  ✗ Some changes after round 5 exceed the threshold — revise discussion.")

In [ ]:
# Confirm the ADOPTED round count on the HELD-OUT TEST set.
# The number of rounds was chosen on validation (above); here the global model at
# the adopted round is evaluated once on the test set, which selection never used.
adopted = keras.models.load_model(adopted_path)
y_pred = np.argmax(adopted.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

acc = accuracy_score(y_true, y_pred) * 100
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
print(f"Held-out TEST metrics for the adopted {ADOPTED_ROUND}-round global model:")
print(f"  Accuracy : {acc:.2f}%")
print(f"  Precision: {prec*100:.2f}%")
print(f"  Recall   : {rec*100:.2f}%")
print(f"  F1-Score : {f1*100:.2f}%")

pd.DataFrame([{'Adopted Round': ADOPTED_ROUND, 'Accuracy (%)': round(acc, 2),
               'Precision (%)': round(prec*100, 2), 'Recall (%)': round(rec*100, 2),
               'F1-Score (%)': round(f1*100, 2)}]).to_csv(
    os.path.join(OUT_DIR, 'adopted_round_test_metrics.csv'), index=False)